In [2]:
df_drivers = (
    spark.read
    .option("header", "true")
    .csv(
        "abfss://Fleet_Logistics_Engineering@onelake.dfs.fabric.microsoft.com/"
        "Fleet_Logistics_Lakehouse.Lakehouse/Files/Landing/drivers.csv"
    )
)

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 4, Finished, Available, Finished, False)

In [3]:
df_drivers.printSchema()

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 5, Finished, Available, Finished, False)

root
 |-- driver_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- hire_date: string (nullable = true)
 |-- termination_date: string (nullable = true)
 |-- license_number: string (nullable = true)
 |-- license_state: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- home_terminal: string (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- cdl_class: string (nullable = true)
 |-- years_experience: string (nullable = true)



In [4]:
print(f"Source records: {df_drivers.count()}")
display(df_drivers.limit(10))


StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 6, Finished, Available, Finished, False)

Source records: 150


SynapseWidget(Synapse.DataFrame, d96cf2dc-dba4-4aee-8531-19b4a9ee7542)

In [5]:
from pyspark.sql import functions as F

# Employment statuses
print("Employment statuses:")
display(
    df_drivers
    .groupBy("employment_status")
    .count()
    .orderBy(F.desc("count"))
)

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 7, Finished, Available, Finished, False)

Employment statuses:


SynapseWidget(Synapse.DataFrame, f44b913a-5e12-43af-84d2-a4450598d338)

In [6]:
# CDL classes
print("CDL classes:")
display(
    df_drivers
    .groupBy("cdl_class")
    .count()
    .orderBy(F.desc("count"))
)

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 8, Finished, Available, Finished, False)

CDL classes:


SynapseWidget(Synapse.DataFrame, 307408c6-9f6a-4d96-bcd4-6350b180cecb)

In [7]:
# License states
print("License states:")
display(
    df_drivers
    .groupBy("license_state")
    .count()
    .orderBy(F.desc("count"))
)

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 9, Finished, Available, Finished, False)

License states:


SynapseWidget(Synapse.DataFrame, eba2f3f2-4752-4913-a526-a3de4266e6e1)

In [8]:
null_driver_ids = (
    df_drivers
    .filter(F.col("driver_id").isNull())
    .count()
)
print(f"NULL driver IDs: {null_driver_ids}")

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 10, Finished, Available, Finished, False)

NULL driver IDs: 0


In [9]:
duplicate_driver_ids = (
    df_drivers
    .groupBy("driver_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print(f"Duplicate driver IDs: {duplicate_driver_ids}")

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 11, Finished, Available, Finished, False)

Duplicate driver IDs: 0


In [10]:
negative_experience = (
    df_drivers
    .filter(F.col("years_experience").cast("int") < 0)
    .count()
)
print(f"Negative years of experience: {negative_experience}")

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 12, Finished, Available, Finished, False)

Negative years of experience: 0


In [11]:
invalid_experience = (
    df_drivers
    .filter(
        F.col("years_experience").isNotNull() &
        F.col("years_experience").cast("int").isNull()
    )
    .count()
)
print(f"Invalid years of experience: {invalid_experience}")

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 13, Finished, Available, Finished, False)

Invalid years of experience: 0


In [12]:
df_driver_dates = (
    df_drivers
    .withColumn("hire_date_check", F.to_date("hire_date"))
    .withColumn("termination_date_check", F.to_date("termination_date"))
    .withColumn("date_of_birth_check", F.to_date("date_of_birth"))
)

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 14, Finished, Available, Finished, False)

In [13]:
display(
    df_driver_dates.select(
        "driver_id",
        "hire_date",
        "hire_date_check",
        "termination_date",
        "termination_date_check",
        "date_of_birth",
        "date_of_birth_check"
    ).limit(10)
)

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3797360e-ced1-4ada-95cd-4bf6115672d5)

In [14]:
df_driver_dates.printSchema()

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 16, Finished, Available, Finished, False)

root
 |-- driver_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- hire_date: string (nullable = true)
 |-- termination_date: string (nullable = true)
 |-- license_number: string (nullable = true)
 |-- license_state: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- home_terminal: string (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- cdl_class: string (nullable = true)
 |-- years_experience: string (nullable = true)
 |-- hire_date_check: date (nullable = true)
 |-- termination_date_check: date (nullable = true)
 |-- date_of_birth_check: date (nullable = true)



In [15]:
invalid_employment_dates = (
    df_driver_dates
    .filter(
        F.col("termination_date_check").isNotNull() &
        (F.col("termination_date_check") < F.col("hire_date_check"))
    )
    .count()
)

print(f"Invalid employment date records: {invalid_employment_dates}")

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 17, Finished, Available, Finished, False)

Invalid employment date records: 0


In [16]:
invalid_birth_dates = (
    df_driver_dates
    .filter(
        F.col("date_of_birth_check").isNotNull() &
        F.col("hire_date_check").isNotNull() &
        (F.col("date_of_birth_check") > F.col("hire_date_check"))
    )
    .count()
)

print(f"Invalid birth/hire date records: {invalid_birth_dates}")

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 18, Finished, Available, Finished, False)

Invalid birth/hire date records: 0


In [17]:
df_drivers_clean = (
    df_driver_dates
    .select(
        "driver_id",
        "first_name",
        "last_name",
        F.col("hire_date_check").alias("hire_date"),
        F.col("termination_date_check").alias("termination_date"),
        "license_number",
        "license_state",
        F.col("date_of_birth_check").alias("date_of_birth"),
        "home_terminal",
        "employment_status",
        "cdl_class",
        F.col("years_experience").cast("int").alias("years_experience")
    )
)

display(df_drivers_clean)

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fbf4f243-31f5-45e4-967f-2a6a73de65cc)

In [18]:
df_drivers_clean.printSchema()

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 20, Finished, Available, Finished, False)

root
 |-- driver_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- termination_date: date (nullable = true)
 |-- license_number: string (nullable = true)
 |-- license_state: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- home_terminal: string (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- cdl_class: string (nullable = true)
 |-- years_experience: integer (nullable = true)



In [19]:
df_drivers_clean = (
    df_drivers_clean
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("drivers.csv"))
)

display(df_drivers_clean.limit(10))

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6cd23c9a-634e-4443-baa6-08f5033a5430)

In [20]:
df_drivers_clean.createOrReplaceTempView("drivers_source")

bronze_drivers_path = "Tables/dbo/bronze_drivers"

print("Source prepared for Bronze MERGE.")

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 22, Finished, Available, Finished, False)

Source prepared for Bronze MERGE.


In [21]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze_drivers (
    driver_id STRING,
    first_name STRING,
    last_name STRING,
    hire_date DATE,
    termination_date DATE,
    license_number STRING,
    license_state STRING,
    date_of_birth DATE,
    home_terminal STRING,
    employment_status STRING,
    cdl_class STRING,
    years_experience INT,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
""")

print("bronze_drivers table is ready.")

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 23, Finished, Available, Finished, False)

bronze_drivers table is ready.


In [22]:
merge_result = spark.sql("""
MERGE INTO bronze_drivers AS target

USING drivers_source AS source

ON target.driver_id = source.driver_id

WHEN MATCHED THEN
    UPDATE SET
        target.first_name = source.first_name,
        target.last_name = source.last_name,
        target.hire_date = source.hire_date,
        target.termination_date = source.termination_date,
        target.license_number = source.license_number,
        target.license_state = source.license_state,
        target.date_of_birth = source.date_of_birth,
        target.home_terminal = source.home_terminal,
        target.employment_status = source.employment_status,
        target.cdl_class = source.cdl_class,
        target.years_experience = source.years_experience,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        driver_id,
        first_name,
        last_name,
        hire_date,
        termination_date,
        license_number,
        license_state,
        date_of_birth,
        home_terminal,
        employment_status,
        cdl_class,
        years_experience,
        ingestion_timestamp,
        source_file
    )

    VALUES (
        source.driver_id,
        source.first_name,
        source.last_name,
        source.hire_date,
        source.termination_date,
        source.license_number,
        source.license_state,
        source.date_of_birth,
        source.home_terminal,
        source.employment_status,
        source.cdl_class,
        source.years_experience,
        source.ingestion_timestamp,
        source.source_file
    )
""")

print("Driver Bronze MERGE completed successfully.")

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 24, Finished, Available, Finished, False)

Driver Bronze MERGE completed successfully.


In [23]:
display(
    spark.sql("""
        SELECT *
        FROM bronze_drivers
        ORDER BY driver_id
        LIMIT 10
    """)
)

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37525204-5fa3-4c5d-b6b8-0a0cc576cdfe)

In [24]:
print(
    "Bronze driver records:",
    spark.sql("SELECT COUNT(*) FROM bronze_drivers").collect()[0][0]
)

StatementMeta(, a00d5d57-61e5-4d8a-8bfa-904696161c04, 26, Finished, Available, Finished, False)

Bronze driver records: 150
